# Extragere date fotbal din football-data.org

Notebook-ul descarcă date pentru o competiție și un sezon prin API-ul football-data.org. Salvează răspunsul brut și fișiere CSV pentru meciuri, goluri, cartonașe și schimbări. Nu antrenează modele.

## 1. Importuri și autentificare

Creează un API token pe football-data.org. Tokenul este introdus ascuns și nu se salvează în notebook.

In [1]:
from getpass import getpass
from dotenv import load_dotenv
from pathlib import Path
import json
import os
import time

import pandas as pd
import requests
from requests.exceptions import HTTPError

API_BASE = "https://api.football-data.org/v4"
OUTPUT_DIR = Path("data/raw/football_data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

load_dotenv()  # citește tokenul din fișierul .env
API_TOKEN = os.getenv("FOOTBALL_DATA_TOKEN", "").strip()
if not API_TOKEN or API_TOKEN == "adauga_tokenul_tau_aici":
    API_TOKEN = getpass("football-data.org API token: ")
if not API_TOKEN.strip():
    raise ValueError("Este necesar un API token football-data.org.")

session = requests.Session()
session.headers.update({
    "X-Auth-Token": API_TOKEN.strip(),
    "X-Unfold-Goals": "true",
    "User-Agent": "IAPD-football-data-project/1.0",
})
print(f"Folder de ieșire: {OUTPUT_DIR.resolve()}")


Folder de ieșire: /Users/laur/Master Data Science/sem1/IAPD/IAPD_project/notebooks/data/raw/football_data


## 2. Selectarea competiției și sezonului

Folosește codul competiției (de exemplu `PL`) și anul de început al sezonului (de exemplu `2024` pentru 2024/25). Competițiile și datele disponibile depind de permisiunile tokenului.

In [ ]:
COMPETITION_CODE = "PL"
SEASON_START_YEAR = 2024
print(f"Competiție: {COMPETITION_CODE}; sezon început în {SEASON_START_YEAR}")


## 3. Funcție pentru cereri API

La HTTP 429 notebook-ul respectă `Retry-After`, apoi reîncearcă.

In [ ]:
def api_get(path: str, params: dict | None = None, max_retries: int = 4) -> dict:
    url = f"{API_BASE}/{path.lstrip('/')}"
    for attempt in range(max_retries + 1):
        response = session.get(url, params=params, timeout=45)
        if response.status_code == 429 and attempt < max_retries:
            wait_seconds = int(response.headers.get("Retry-After", 60))
            print(f"Limită de cereri atinsă; reîncerc peste {wait_seconds} secunde.")
            time.sleep(wait_seconds)
            continue
        try:
            response.raise_for_status()
        except HTTPError as exc:
            try:
                details = response.json()
            except ValueError:
                details = response.text[:500]
            raise RuntimeError(f"API HTTP {response.status_code}: {details}") from exc
        return response.json()
    raise RuntimeError("Nu s-a primit un răspuns valid după reîncercări.")


## 4. Verificarea competițiilor disponibile

In [ ]:
competitions_payload = api_get("competitions/")
competitions = pd.json_normalize(competitions_payload.get("competitions", []), sep=".")
columns = [c for c in ["code", "name", "area.name"] if c in competitions.columns]
display(competitions[columns])
if COMPETITION_CODE not in set(competitions.get("code", pd.Series(dtype=str)).dropna()):
    print("Atenție: codul nu apare în lista disponibilă acestui token.")


## 5. Extragerea meciurilor

Răspunsul JSON complet se păstrează pentru trasabilitate.

In [ ]:
matches_payload = api_get(
    f"competitions/{COMPETITION_CODE}/matches",
    params={"season": SEASON_START_YEAR},
)
with (OUTPUT_DIR / f"{COMPETITION_CODE}_{SEASON_START_YEAR}_matches_raw.json").open("w", encoding="utf-8") as f:
    json.dump(matches_payload, f, ensure_ascii=False, indent=2)

matches = pd.json_normalize(matches_payload.get("matches", []), sep=".")
matches.to_csv(OUTPUT_DIR / f"{COMPETITION_CODE}_{SEASON_START_YEAR}_matches.csv", index=False)
print(f"Meciuri extrase: {len(matches):,}")
display(matches.head())


## 6. Goluri, cartonașe și schimbări

Aceste tabele se extrag din răspunsul meciurilor. Câmpurile disponibile pot varia în funcție de competiție și permisiunile API.

In [ ]:
def flatten_match_events(match_rows: list[dict], field: str) -> pd.DataFrame:
    records = []
    for match in match_rows:
        for event in match.get(field, []) or []:
            records.append({
                "match_id": match.get("id"),
                "competition": (match.get("competition") or {}).get("name"),
                "season_start": (match.get("season") or {}).get("startDate"),
                "utcDate": match.get("utcDate"),
                **event,
            })
    return pd.json_normalize(records, sep=".")

match_rows = matches_payload.get("matches", [])
goals = flatten_match_events(match_rows, "goals")
bookings = flatten_match_events(match_rows, "bookings")
substitutions = flatten_match_events(match_rows, "substitutions")

goals.to_csv(OUTPUT_DIR / f"{COMPETITION_CODE}_{SEASON_START_YEAR}_goals.csv", index=False)
bookings.to_csv(OUTPUT_DIR / f"{COMPETITION_CODE}_{SEASON_START_YEAR}_bookings.csv", index=False)
substitutions.to_csv(OUTPUT_DIR / f"{COMPETITION_CODE}_{SEASON_START_YEAR}_substitutions.csv", index=False)
print({"goluri": len(goals), "cartonașe": len(bookings), "schimbări": len(substitutions)})


## 7. Salvarea competițiilor și rezumat

In [ ]:
competitions.to_json(
    OUTPUT_DIR / "competitions.json",
    orient="records",
    force_ascii=False,
    indent=2,
)
print("Extragerea s-a încheiat. Fișierele sunt în:", OUTPUT_DIR.resolve())


### Limite ale datelor

API-ul oferă în principal date despre competiții, meciuri și goluri. Line-up-urile și evenimentele suplimentare pot depinde de competiție și planul API. Nu presupune că minutele, șuturile sau cartonașele sunt disponibile pentru fiecare meci; verifică lipsurile înainte de analiză.